In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Jalankan litar pertama anda pada perkakasan

<Accordion>
<AccordionItem title="Versi pakej">

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan menggunakan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Contoh ini mengandungi dua bahagian. Anda akan terlebih dahulu mencipta program kuantum "Hello world" mudah dan menjalankannya pada unit pemprosesan kuantum (QPU). Oleh kerana penyelidikan kuantum sebenar memerlukan program yang jauh lebih teguh, dalam bahagian kedua ([Skala ke bilangan qubit yang besar](#scale-to-large-numbers-of-qubits)), anda akan menskalakan program mudah tersebut ke tahap utiliti.

## Pasang dan sahkan diri
1. Jika anda belum memasang Qiskit, cari arahan dalam panduan [Quickstart](/guides/quick-start).

    - Pasang Qiskit Runtime untuk menjalankan kerja pada perkakasan kuantum:

        ```bash
        pip install qiskit-ibm-runtime
        ```

    - Sediakan persekitaran untuk menjalankan Jupyter notebooks secara tempatan:

        ```bash
        pip install jupyter
        ```

2. Sediakan pengesahan anda untuk akses ke perkakasan kuantum melalui [Pelan Terbuka](/guides/plans-overview#open-plan) percuma.

    (Jika anda menerima jemputan melalui e-mel untuk menyertai akaun, ikuti [langkah-langkah untuk pengguna yang dijemput](/guides/cloud-setup-invited) sebaliknya.)

    - Pergi ke [IBM Quantum Platform](https://quantum.cloud.ibm.com/) untuk log masuk atau membuat akaun.
         > **Note:** Jika anda menyambung melalui pelayan proksi, anda mesti menggunakan Qiskit Runtime v0.44.0 atau lebih baharu.
    - Jana kunci API anda (juga dipanggil *token API*) pada [papan pemuka](https://quantum.cloud.ibm.com/), kemudian salinnya ke lokasi yang selamat.
    - Pergi ke halaman [Instances](https://quantum.cloud.ibm.com/instances) dan cari instance yang ingin anda gunakan. Arahkan tetikus ke atas CRN-nya dan klik untuk menyalinnya.

    - Simpan kelayakan anda secara tempatan dengan kod ini:

        ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        QiskitRuntimeService.save_account(
            token="<your-api-key>", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
            instance="<CRN>", # Optional
        )
        ```

3. Kini anda boleh menggunakan kod Python ini bila-bila masa anda mahu mengesahkan diri ke Qiskit Runtime Service:
    ```python
    from qiskit_ibm_runtime import QiskitRuntimeService

    # Run every time you need the service
    service = QiskitRuntimeService()
    ```
> **Info:** Jika anda menggunakan komputer awam atau persekitaran lain yang tidak selamat, ikuti [arahan pengesahan manual](/guides/cloud-setup-untrusted) sebaliknya untuk memastikan kelayakan pengesahan anda selamat.
## Cipta dan jalankan program kuantum mudah
Empat langkah untuk menulis program kuantum menggunakan corak Qiskit adalah:

1.  Petakan masalah ke format asli kuantum.

2.  Optimumkan litar dan operator.

3.  Laksanakan menggunakan fungsi primitif kuantum.

4.  Analisis keputusan.

### Langkah 1. Petakan masalah ke format asli kuantum
Dalam program kuantum, *litar kuantum* ialah format asli untuk mewakili arahan kuantum, dan *operator* mewakili boleh cerap yang perlu diukur. Apabila mencipta litar, anda biasanya akan mencipta objek [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) baharu, kemudian menambah arahan ke dalamnya secara berurutan.
Sel kod berikut mencipta litar yang menghasilkan *keadaan Bell,* iaitu keadaan di mana dua qubit terbelit sepenuhnya antara satu sama lain.

> **Note:** SDK Qiskit menggunakan penomboran bit LSb 0 di mana digit ke-$n$ mempunyai nilai $1 \ll n$ atau $2^n$. Untuk butiran lanjut, lihat topik [Susunan bit dalam SDK Qiskit](/guides/bit-ordering).

In [1]:
# This cell is hidden from users.  It hides several unnecessary warnings.

import warnings
import logging

warnings.filterwarnings("ignore", message=".*Instance was not set*")
warnings.filterwarnings("ignore", message=".*loading instance*")
warnings.filterwarnings("ignore", message=".*using instance*")

# This cell is hidden from users.  It hides several unnecessary warnings.


class IgnoreSpecificMessages(logging.Filter):
    def filter(self, record):
        for text in [
            "Instance was not set",
            "Loading default saved account",
            "Loading instance",
            "Instance was not set",
            "using instance",
        ]:
            if text in record.getMessage():
                return False
        return True


logger = logging.getLogger("qiskit_ibm_runtime")
logger.addFilter(IgnoreSpecificMessages())

for handler in logger.handlers:
    handler.addFilter(IgnoreSpecificMessages())

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from matplotlib import pyplot as plt
# Uncomment the next line if you want to use a simulator:
# from qiskit_ibm_runtime.fake_provider import FakeBelemV2


# Create a new circuit with two qubits
qc = QuantumCircuit(2)

# Add a Hadamard gate to qubit 0
qc.h(0)

# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)

# Return a drawing of the circuit using MatPlotLib ("mpl").
# These guides are written by using Jupyter notebooks, which
# display the output of the last line of each cell.
# If you're running this in a script, use `print(qc.draw())` to
# print a text drawing.
qc.draw("mpl")

<Image src="../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg" alt="Output of the previous code cell" />

> **Note:** Di sini, sesuatu seperti operator `ZZ` ialah singkatan untuk hasil darab tensor $Z\otimes Z$, yang bermaksud mengukur Z pada qubit 1 dan Z pada qubit 0 bersama-sama, dan mendapatkan maklumat tentang korelasi antara qubit 1 dan qubit 0. Nilai jangkaan seperti ini juga biasanya ditulis sebagai $\langle Z_1 Z_0 \rangle$.
> 
> Jika keadaan terbelit, maka pengukuran $\langle Z_1 Z_0 \rangle$ sepatutnya berbeza daripada pengukuran $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$. Untuk keadaan terbelit khusus yang dicipta oleh litar kami yang diterangkan di atas, pengukuran $\langle Z_1 Z_0 \rangle$ sepatutnya 1 dan pengukuran $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$ sepatutnya sifar.
<span id="optimize"></span>

### Langkah 2. Optimumkan litar dan operator

Apabila melaksanakan litar pada peranti, adalah penting untuk mengoptimumkan set arahan yang terkandung dalam litar dan meminimumkan kedalaman keseluruhan (secara kasar bilangan arahan) litar. Ini memastikan anda mendapat hasil terbaik yang mungkin dengan mengurangkan kesan ralat dan hingar. Selain itu, arahan litar mesti mematuhi [Seni Bina Set Arahan (ISA)](/guides/transpile#instruction-set-architecture) peranti backend dan mesti mempertimbangkan get asas dan ketersambungan qubit peranti.

Kod berikut memulakan peranti nyata untuk menghantar kerja dan mengubah litar serta operator supaya sepadan dengan ISA backend tersebut. Ia memerlukan anda sudah [menyimpan kelayakan anda](/guides/cloud-setup)

In [3]:
# Set up six different observables.

observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg)

### Langkah 3. Laksanakan menggunakan primitif kuantum

Komputer kuantum boleh menghasilkan keputusan rawak, jadi anda biasanya mengumpul sampel output dengan menjalankan litar berkali-kali. Anda boleh menganggar nilai boleh cerap menggunakan kelas `Estimator`. `Estimator` ialah salah satu daripada dua [primitif](/guides/get-started-with-estimator); yang satu lagi ialah `Sampler`, yang boleh digunakan untuk mendapatkan data daripada komputer kuantum. Objek-objek ini mempunyai kaedah `run()` yang melaksanakan pemilihan litar, boleh cerap, dan parameter (jika berkenaan), menggunakan [blok bersatu primitif (PUB)](/guides/get-started-with-sampler).

In [4]:
service = QiskitRuntimeService()

backend = service.least_busy(simulator=False, operational=True)

# Convert to an ISA circuit and layout-mapped observables.
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

isa_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg" alt="Output of the previous code cell" />

### Step 3. Execute using the quantum primitives

Quantum computers can produce random results, so you usually collect a sample of the outputs by running the circuit many times. You can estimate the value of the observable by using the `Estimator` class. `Estimator` is one of two [primitives](/docs/guides/get-started-with-estimator); the other is `Sampler`, which can be used to get data from a quantum computer.  These objects possess a `run()` method that executes the selection of circuits, observables, and parameters (if applicable), using a [primitive unified bloc (PUB)](/docs/guides/get-started-with-sampler).

In [5]:
# Construct the Estimator instance.

estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

# One pub, with one circuit to run against five different observables.
job = estimator.run([(isa_circuit, mapped_observables)])

# Use the job ID to retrieve your job data later
print(f">>> Job ID: {job.job_id()}")

>>> Job ID: d8286mfoha1c73bl8hrg


Selepas kerja dihantar, anda boleh menunggu sehingga kerja selesai dalam instance Python semasa anda, atau gunakan `job_id` untuk mendapatkan semula data pada masa yang lain. (Lihat [bahagian tentang mendapatkan semula kerja](/guides/monitor-job#retrieve-job-results-at-a-later-time) untuk butiran.)

Selepas kerja selesai, periksa outputnya melalui atribut `result()` kerja.

In [6]:
# This is the result of the entire submission.  You submitted one Pub,
# so this contains one inner result (and some metadata of its own).
job_result = job.result()

# This is the result from our single pub, which had six observables,
# so contains information on all six.
pub_result = job.result()[0]

In [7]:
# Check there are six observables.
# If not, edit the comments in the previous cell and update this test.
assert len(pub_result.data.evs) == 6

<Admonition type="note" title="Alternative: run the example using a simulator">
  When you run your quantum program on a real device, your workload must wait in a queue before it runs. To save time, you can instead use the following code to run this small workload on the [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) with the Qiskit Runtime local testing mode. Note that this is only possible for a small circuit. When you scale up in the next section, you will need to use a real device.

  ```python

  # Use the following code instead if you want to run on a simulator:

  from qiskit_ibm_runtime.fake_provider import FakeBelemV2
  backend = FakeBelemV2()
  estimator = Estimator(backend)

  # Convert to an ISA circuit and layout-mapped observables.

  pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
  isa_circuit = pm.run(qc)
  mapped_observables = [
      observable.apply_layout(isa_circuit.layout) for observable in observables
  ]

  job = estimator.run([(isa_circuit, mapped_observables)])
  result = job.result()

  # This is the result of the entire submission.  You submitted one Pub,
  # so this contains one inner result (and some metadata of its own).

  job_result = job.result()

  # This is the result from our single pub, which had five observables,
  # so contains information on all five.

  pub_result = job.result()[0]
  ```
</Admonition>

### Step 4. Analyze the results

The analyze step is typically where you might post-process your results using, for example, measurement error mitigation or zero noise extrapolation (ZNE). You might feed these results into another workflow for further analysis or prepare a plot of the key values and data. In general, this step is specific to your problem.  For this example, plot each of the expectation values that were measured for our circuit.

The expectation values and standard deviations for the observables you specified to Estimator are accessed through the job result's `PubResult.data.evs` and `PubResult.data.stds` attributes. To obtain the results from Sampler, use the `PubResult.data.meas.get_counts()` function, which will return a `dict` of measurements in the form of bitstrings as keys and counts as their corresponding values. For more information, see the [Sampler quickstart](/docs/guides/get-started-with-sampler) guide.

In [8]:
# Plot the result

values = pub_result.data.evs

errors = pub_result.data.stds

# plotting graph
plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Values")
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg" alt="Output of the previous code cell" />

:::note[Alternatif: jalankan contoh menggunakan simulator]

  Apabila anda menjalankan program kuantum pada peranti nyata, beban kerja anda mesti menunggu dalam giliran sebelum berjalan. Untuk menjimatkan masa, anda boleh menggunakan kod berikut untuk menjalankan beban kerja kecil ini pada [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) dengan mod ujian tempatan Qiskit Runtime. Perhatikan bahawa ini hanya mungkin untuk litar yang kecil. Apabila anda menskalakan dalam bahagian seterusnya, anda perlu menggunakan peranti nyata.

  ```python

  # Use the following code instead if you want to run on a simulator:

  from qiskit_ibm_runtime.fake_provider import FakeBelemV2
  backend = FakeBelemV2()
  estimator = Estimator(backend)

  # Convert to an ISA circuit and layout-mapped observables.

  pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
  isa_circuit = pm.run(qc)
  mapped_observables = [
  observable.apply_layout(isa_circuit.layout) for observable in observables
  ]

  job = estimator.run([(isa_circuit, mapped_observables)])
  result = job.result()

  # This is the result of the entire submission.  You submitted one Pub,
  # so this contains one inner result (and some metadata of its own).

  job_result = job.result()

  # This is the result from our single pub, which had five observables,
  # so contains information on all five.

  pub_result = job.result()[0]
  ```

:::
### Langkah 4. Analisis keputusan
Langkah analisis biasanya adalah tempat anda mungkin memproses semula keputusan menggunakan, contohnya, pengurangan ralat pengukuran atau ekstrapolasi hingar sifar (ZNE). Anda mungkin memasukkan keputusan ini ke dalam aliran kerja lain untuk analisis lebih lanjut atau menyediakan plot nilai dan data utama. Secara amnya, langkah ini khusus kepada masalah anda. Untuk contoh ini, plotkan setiap nilai jangkaan yang diukur untuk litar kami.

Nilai jangkaan dan sisihan piawai untuk boleh cerap yang anda nyatakan kepada Estimator diakses melalui atribut `PubResult.data.evs` dan `PubResult.data.stds` hasil kerja. Untuk mendapatkan keputusan daripada Sampler, gunakan fungsi `PubResult.data.meas.get_counts()`, yang akan mengembalikan `dict` pengukuran dalam bentuk bitstring sebagai kunci dan kiraan sebagai nilai yang sepadan. Untuk maklumat lanjut, lihat panduan [Mula dengan Sampler](/guides/get-started-with-sampler).

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg)

Perhatikan bahawa untuk qubit 0 dan 1, nilai jangkaan bebas bagi X dan Z adalah 0, manakala korelasi (`XX` dan `ZZ`) adalah 1. Ini adalah tanda pengenal jeratan kuantum.

In [9]:
# Make sure the results follow the claim from the previous markdown cell.
# This can happen when the device occasionally behaves strangely. If this cell
# fails, you may just need to run the notebook again.

_results = {obs: val for obs, val in zip(observables_labels, values)}
for _label in ["IZ", "IX", "ZI", "XI"]:
    assert abs(_results[_label]) < 0.2
for _label in ["XX", "ZZ"]:
    assert _results[_label] > 0.8

## Skala ke bilangan qubit yang besar
Dalam pengkomputeran kuantum, kerja berskala utiliti adalah penting untuk membuat kemajuan dalam bidang ini. Kerja sedemikian memerlukan pengiraan dilakukan pada skala yang jauh lebih besar; bekerja dengan litar yang mungkin menggunakan lebih daripada 100 qubit dan lebih daripada 1000 get. Contoh ini menunjukkan cara anda boleh menyelesaikan kerja berskala utiliti pada QPU IBM&reg; dengan mencipta dan menganalisis keadaan GHZ 100 qubit. Ia menggunakan aliran kerja corak Qiskit dan berakhir dengan mengukur nilai jangkaan $\langle Z_0 Z_i \rangle$ untuk setiap qubit.

### Langkah 1. Petakan masalah
Tulis fungsi yang mengembalikan `QuantumCircuit` yang menyediakan keadaan GHZ $n$-qubit (pada dasarnya keadaan Bell yang dilanjutkan), kemudian gunakan fungsi tersebut untuk menyediakan keadaan GHZ 100 qubit dan kumpulkan boleh cerap yang perlu diukur.

In [10]:
def get_qc_for_n_qubit_GHZ_state(n: int) -> QuantumCircuit:
    """This function will create a qiskit.QuantumCircuit (qc) for an n-qubit GHZ state.

    Args:
        n (int): Number of qubits in the n-qubit GHZ state

    Returns:
        QuantumCircuit: Quantum circuit that generate the n-qubit GHZ state, assuming all qubits start in the 0 state
    """
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
    else:
        raise Exception("n is not a valid input")
    return qc


# Create a new circuit with 100 qubits in the GHZ state
n = 100
qc = get_qc_for_n_qubit_GHZ_state(n)

Seterusnya, petakan kepada operator yang diminati. Contoh ini menggunakan operator `ZZ` antara qubit untuk mengkaji tingkah laku apabila ia semakin berjauhan. Nilai jangkaan antara qubit yang berjauhan yang semakin tidak tepat (rosak) akan mendedahkan tahap hingar yang hadir.

In [11]:
# ZZII...II, ZIZI...II, ... , ZIII...IZ
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]

operators = [SparsePauliOp(operator) for operator in operator_strings]

### Step 2. Optimize the problem for execution on quantum hardware

The following code transforms the circuit and observables to match the backend's ISA. It requires that you have already [saved your credentials](/docs/guides/cloud-setup)

In [12]:
service = QiskitRuntimeService()

backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

isa_circuit = pm.run(qc)
isa_operators_list = [op.apply_layout(isa_circuit.layout) for op in operators]

### Langkah 2. Optimumkan masalah untuk pelaksanaan pada perkakasan kuantum
Kod berikut mengubah litar dan operator supaya sepadan dengan ISA backend. Ia memerlukan anda sudah [menyimpan kelayakan anda](/guides/cloud-setup)

In [13]:
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

# Create an Estimator object
estimator = Estimator(backend, options=options)

In [14]:
# Submit the circuit to Estimator
job = estimator.run([(isa_circuit, isa_operators_list)])
job_id = job.job_id()
print(job_id)

d828aeo0bvlc73d1vs20


### Step 4. Post-process results

After the job completes, plot the results and notice that $\langle Z_0 Z_i \rangle$ decreases with increasing $i$, even though in an ideal simulation all $\langle Z_0 Z_i \rangle$ should be 1.

In [15]:
# data
data = list(range(1, len(operators) + 1))  # Distance between the Z operators
result = job.result()[0]
values = result.data.evs  # Expectation value at each Z operator.
values = [
    v / values[0] for v in values
]  # Normalize the expectation values to evaluate how they decay with distance.

# plotting graph
plt.plot(data, values, marker="o", label="100-qubit GHZ state")
plt.xlabel("Distance between qubits $i$")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle $")
plt.legend()
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/de91ebd0-0.svg" alt="Output of the previous code cell" />

The previous plot shows that as the distance between qubits increases, the signal decays because of the presence of noise.

## Next steps

<Admonition type="tip" title="Recommendations">
  -   Try one of these tutorials:
      - [Ground-state energy estimation of the Heisenberg chain with VQE](/docs/tutorials/spin-chain-vqe)
      - Solve optimization problems using [QAOA](/docs/tutorials/quantum-approximate-optimization-algorithm)
      - Train [quantum kernel](/docs/tutorials/quantum-kernel-training) models for machine learning tasks
  - Find detailed installation instructions in the [Install Qiskit](/docs/guides/install-qiskit) guide.
  - If you prefer not to install Qiskit locally, read about options to use Qiskit in an [online development environment](/docs/guides/online-lab-environments).
  - To save multiple account credentials or to specify other account options, see detailed instructions in the [Save your login credentials](/docs/guides/save-credentials#save-your-access-credentials) guide.
</Admonition>